# VEP Research II — Updated Model Landscape (July 2026)
## Other VEP architectures we could try for nAChR GOF/LOF prediction

*Date: 2026-07-09*

**Relationship to `vep_research.ipynb` (2026-06-14):** that notebook already answered "is VEP just the algorithm?" (no — it's a 5-part pipeline) and mapped the field into six families, landing on **funNCion, LoGoFunc, PreMode, MissION** as the closest prior work on our exact task (GOF vs LOF *direction*, not generic pathogenicity). This notebook does **not repeat that** — it adds what's new since then, from two sources:

1. **The `VEP papers/` folder** — 16 real papers already collected in this repo, read in full for this notebook (a benchmarking/critical-assessment/guidelines cluster, plus a genuine ion-channel find: a 2002 mechanistic paper on nAChR congenital myasthenic syndrome mutations).
2. **A fresh literature scan** (July 2026) that (a) resolves the open question from the June notebook — *does MissION's 47-gene panel include nAChR?* — and (b) surfaces newer ensemble/benchmark work (Livesey & Marsh 2025, *Genome Biology*) that re-ranks the whole field.

**Headline update:** nAChR-specific GOF/LOF prediction is still an open gap (Section 1). The field's best current *general* VEPs are no longer just AlphaMissense/ESM — a 2025 pairwise-benchmark crowns an **ensemble of EVE + ESM-1v + structure ("CPT-1")** as the new top performer, with several cheaper population-free alternatives (GEMME/iGEMME/ESCOTT) close behind (Section 2). Two more directly-relevant papers turned up in our own `VEP papers/` folder (Section 3). Section 4 updates the evaluation playbook with a circularity taxonomy and a fairer ranking method we should copy.

---
## 1. Resolved: does MissION cover nAChR? — No, and neither does PreMode

The June notebook flagged this as the deciding question for whether a nAChR-specific model is a genuine contribution or a re-run of existing work. Checked now:

- **MissION** (Martens et al., medRxiv 2025.10.16.25337735 → published 2026 in *Journal of Human Genetics*, DOI [10.1038/s10038-026-01484-9](https://www.nature.com/articles/s10038-026-01484-9)) trains on 3,176 GOF/LOF variants across **47 genes that are predominantly voltage-gated Na⁺/K⁺/Ca²⁺ channels**. No CHRNA/CHRNB/CHRND/CHRNE/CHRNG (or any Cys-loop / pentameric ligand-gated receptor gene) appears in the fetchable text or figures. The paper itself notes many ion-channel functional-effect studies "exclud[e] accessory subunits" — consistent with ligand-gated pentamers being out of scope. An online tool now exists at [synaptica.nl/variant-interpreter](https://www.synaptica.nl/variant-interpreter) with >600k precomputed ion-channel variant scores — worth checking later whether nAChR ever gets added, but as of this scan it has not.
- **PreMode** (now version-of-record in *Nature Communications*, Aug 2025, [10.1038/s41467-025-62318-4](https://www.nature.com/articles/s41467-025-62318-4)) fine-tunes gene-by-gene on ClinVar/ClinGen genes with abundant functional data — reported focus is cardiac/neuro channels and kinases. No evidence of CHRN gene fine-tuning.
- Closest *adjacent* precedent found: two 2023–2024 papers apply **AlphaMissense (+ Rhapsody)** to **GABA_A receptor** subunits (GABRA1/B2/B3/G2) — our nAChR's fellow **Cys-loop receptor**. One follow-up (PMC11648851) found many GABA_A epilepsy variants converge on **GOF via increased agonist sensitivity**, a mechanistic parallel worth keeping in mind for nAChR (Section 3.4).
- A July-2025 preprint, **"Deciphering gain-of-function from loss-of-function variants with AlphaMissense: a case study with... PIEZO1"** (bioRxiv 2025.07.03.662957, Pillai et al., code at `github.com/Joshua-Pillai/PIEZO1`), is not a trained GOF/LOF classifier either — it's a **methodological template**: take an off-the-shelf pathogenicity score (AlphaMissense/CADD/EVE), split known GOF vs LOF variants, and show the scores + structural location separate the two groups. This is a cheap, no-training pattern we could replicate on nAChR immediately as a sanity baseline before building anything supervised (Section 3.3).

**Conclusion — unchanged from June, now confirmed with evidence:** a nAChR-specific GOF/LOF predictor combining our curated data, the structural features already in the pipeline (`structural.py`), and the newer feature sources below (`features.ipynb`) remains a defensible, non-duplicative contribution.

---
## 2. The field re-ranked: Livesey & Marsh 2025 (*Genome Biology*) pairwise benchmark

This is the single most useful new reference for "what else is out there" — a systematic pairwise win-rate comparison of **97 VEPs** against deep-mutational-scanning (DMS) data across 36 proteins. It doesn't predict GOF/LOF direction (still a pathogenicity/fitness benchmark, per the family map in the June notebook), but it reshuffles which *general* VEPs are worth borrowing as feature sources or baselines.

| Model | Rank (of 97) | Win rate | Type | Verdict for us |
|---|---|---|---|---|
| **CPT-1** | 1 | 92.8% | Ensemble: EVE + ESM-1v + AlphaFold/ProteinMPNN structure + conservation | **Adopt as ensemble-design template** — its recipe (unsupervised generative + pLM + structure) is close to what a nAChR ensemble should look like |
| **AlphaMissense** | 2 | 90.7% | Population-tuned (fine-tuned with allele frequency as weak labels) | **Keep as primary baseline feature**, but tag it "population-tuned" not fully unsupervised (circularity bookkeeping, Section 4) |
| **ESCOTT** | 3 | — | Population-free; evolution/epistasis + structural context of the mutated residue | Candidate lightweight feature source |
| **iGEMME** | 5 | — | Population-free; phylogeny-derived substitution likelihood, scales to large proteins | Cheapest good option — no pLM download needed |
| **GEMME** | 6 | — | Predecessor of iGEMME, same family | Same as above |
| **popEVE** | 4 | — | Hybrid ESM-1v + EVE **with gene-level calibration** | The gene-level calibration trick is directly relevant — our subunits have wildly uneven mutation counts (18 for CHRNA6 vs. much more for CHRNA1/CHRNE) |
| **SaProt** | 7 | — | Structure-aware pLM (folds AlphaFold structure tokens into the sequence-model vocabulary) | Candidate embedding source; fuses structure+sequence more natively than concatenating separate features |
| **VARITY** | (DMM 2022 review, not in the 97-set table above) | — | Supervised meta-predictor combining heterogeneous training sources (incl. functional assays) with data-quality weighting | **Adopt as benchmark** — its "weight noisy heterogeneous sources" design matches our messy, multi-lab-sourced 351-variant dataset well |
| TranceptEVE, MutFormer, gMVP, MSA-Transformer, CARP, MetaRNN, PHACT/PHACTboost, mvPPT, SNPred, MutScore | — | — | Various supervised/unsupervised entrants in the same 97-model table | Cite for context only; no individual verdict warranted |

**Two training-technique / framing ideas from the same reading pass (not classifiers to benchmark, but tricks to borrow):**

- **MTBAN** ("An enhanced variant effect predictor based on a deep generative model and the Born-Again Networks") applies **self-distillation**: retrain the same generative-VEP architecture on its own prior generation's soft outputs to improve calibration *without new labels*. Directly applicable to a small-data problem like ours if we ever fine-tune an ESM/EVE-style base model on nAChR data.
- **Envision** (Gray et al. 2018) trains a random forest directly on **DMS data** across 9 proteins to predict a **continuous functional score**, not a binary label — using PSSM conservation + biophysical/structural features. This "predict a continuous fitness-like score, not a binary class" framing is arguably closer to what GOF/LOF *magnitude* should look like than standard binary VEPs. (Caveat: Livesey & Marsh flag Envision as one of only 5 VEPs directly DMS-trained — must be excluded from any DMS-based benchmarking we do to avoid circularity.)

**Lower-priority, cite-only findings from the same reading pass:**
- **APF2** — ensemble/meta-predictor for *pharmacogenomic* variant effect (drug-response variants). Different target than pathogenicity/GOF-LOF; methodology (ensembling) is generic but the framing doesn't transfer.
- **AlphaGenome** — DeepMind's DNA-to-regulatory-function model (splicing/expression/chromatin), not a missense-effect predictor. Relevant only if we ever need regulatory/splice-variant scoring for CHRN genes.
- **Ensembl VEP** (the tool itself, both the original *Genome Biology* paper and the 2021 tutorial) — variant **annotation infrastructure** (HGVS, MANE transcripts, SO consequence terms, GA4GH VRS, ClinGen Allele Registry IDs), not a predictive model. **Adopt as our annotation/data-representation backbone** for reproducibility, not as a classifier.
- **"VEP Finder" systematic review** (118 tools cataloged) — notes ESM-1v/EVE/DeepSequence were excluded from their review for not meeting strict "accepts raw variant input" criteria, despite being top performers elsewhere. Useful mainly as a reminder to standardize our own model's input/output conventions if we ever publish it.

---
## 3. What was already sitting in our own `VEP papers/` folder

Four papers in the folder turned out to be directly on-topic and were not yet folded into the June notebook.

### 3.1 Ion-channel phenotypic ML paper — a genuinely new feature category (HPO kernels)

*"Predicting functional effects of ion channel variants using new phenotypic machine learning methods."* This is the most directly relevant paper in the batch: it classifies ion-channel variants into functional-effect classes (i.e., our exact GOF/LOF-style target), not generic pathogenicity. Its core contribution is a **multiple-kernel-learning (MKL)** framework that adds a **Human Phenotype Ontology (HPO) semantic-similarity kernel** (via Jaccard, Lin, and Resnik similarity measures) on top of sequence/structural features.

**Why this matters for us:** none of funNCion / LoGoFunc / PreMode / MissION use phenotype-derived similarity as an input. nAChR mutations are richly phenotype-annotated (congenital myasthenic syndrome, ADNFLE/epilepsy, nicotine-dependence GWAS) — an HPO-similarity kernel is a concrete, currently-missing feature category. See `features.ipynb` §3.7 for the practical recipe.

### 3.2 Voltage-gated K⁺ channel multi-task learning (EBioMedicine 2022) — deeper dive

Already namechecked in the June notebook; re-read in full here. Its actual architecture is **multi-task learning (MTL) with a phylogeny/taxonomy-derived task-similarity kernel** (the Widmer et al. / Jacob & Vert MTL-kernel formalism): instead of one model per gene, it jointly trains classifiers across multiple related K⁺-channel genes, using a kernel that encodes how similar two channel genes are (by sequence/phylogenetic distance) so that data-rich genes lend statistical strength to data-poor ones.

**Why this is a strong direct template for nAChR:** CHRNA1–10 / CHRNB1–4 / CHRND/E/G is exactly this kind of unevenly-sampled paralog family — see the subunit imputation table in `NOTES2.ipynb` §10 (CHRNA6: 18 imputed, CHRNA2: 9, CHRNA5: 5, CHRNB3: 4, CHRNA9/A10: 2 each — the same subunits that lack PDB structures also tend to have the least mutation data). A phylogeny-weighted MTL kernel across subunits is a concrete way to borrow signal from data-rich CHRNA1/CHRNE (both CMS-relevant, well-studied) for data-poor neuronal subunits.

### 3.3 The PIEZO1 AlphaMissense case study — a cheap sanity-check template

Covered in Section 1 above. Concretely reusable pattern: take AlphaMissense (or CADD/EVE) scores for our known GOF and LOF nAChR variants, check whether the two groups separate on score + structural location, **before** building anything supervised. If they don't separate at all, that's an important negative result about how much "pathogenicity" alone can tell us about *direction*.

### 3.4 The Congenital Myasthenic Syndrome paper (Engel, Ohno & Sine, *Molecular Neurobiology* 2002) — mechanistic ground truth, found by accident

One PDF in the folder was misleadingly named "PDF.js viewer.pdf" — it is actually this CMS review, and it turned out to be one of the most useful reads. It documents nAChR mutations with clear, opposite, and *mechanistically explained* directions:

- **LOF ("fast-channel syndrome")**: mutations that reduce ACh binding affinity or gating efficiency — concentrated in the **extracellular ligand-binding domain** (e.g., ε-subunit binding-site loop mutations) — reduced synaptic current.
- **GOF ("slow-channel syndrome")**: mutations that **prolong channel-open burst duration** — concentrated in the **M2 pore-lining transmembrane helix** (e.g., εL269F, αV249F) — excessive/prolonged cation influx, endplate myopathy from Ca²⁺ overload.

**Why this matters:** it gives structurally-grounded, mechanistically-validated examples where **mutation location (binding-site loop vs. M2 pore helix) alone strongly predicts GOF vs. LOF direction.** This directly motivates the domain/region-indicator feature already sketched in `NOTES2.ipynb` §11.5, and can double as a small held-out mechanistic sanity-check set (not just training data) for whatever model we build. See `features.ipynb` §3.6.

*(Aside, confirmed and excluded: "VEP estimation of visual acuity a systematic review.pdf" in the same folder is genuinely about ophthalmological Visual Evoked Potentials — an unrelated field that happens to share the acronym "VEP." Not relevant here.)*

---
## 4. Updated evaluation playbook

The benchmarking/critical-assessment/guidelines cluster in `VEP papers/` (Livesey & Marsh 2025; "Critical assessment of missense VEPs on disease-relevant data"; "Benchmarking computational VEPs by their ability to infer human traits"; "Guidelines for releasing a variant effect predictor") converges on several concrete practices to fold into our own evaluation, on top of what `vep_research.ipynb` §6 already said (grouped CV, right baselines, imbalance-aware metrics).

1. **Gene-level circularity is our dominant risk, not variant-level.** With only 16 subunit genes and very uneven per-gene counts, our existing subunit-stratified thinking should become an explicit **leave-one-subunit-out CV** check, matching how the K⁺-channel MTL paper validates cross-gene generalization (§3.2).
2. **Tag every feature/model input by circularity tier**: *clinical-trained* (ClinVar/HGMD-trained — highest risk if reused as a feature and then evaluated on clinical variants), *population-tuned* (exposed to allele frequency, e.g. AlphaMissense — moderate risk, especially for rare benign nAChR variants), *population-free* (GEMME/iGEMME/ESCOTT, EVE — safest as an independent feature). Carry this tag into the feature table in `features.ipynb`.
3. **Prefer pairwise win-rate ranking over raw mean correlation** when we eventually compare multiple candidate feature sets/models on held-out data — it avoids rewarding a method for only covering an "easy" subset of variants (Livesey & Marsh's specific fix, directly reusable once we have ≥2 candidate scores per variant).
4. **DMS/MAVE data is the cleanest circularity-free benchmark substrate** — but note upfront: no nAChR-specific DMS dataset exists in the Livesey & Marsh 36-protein set or (as far as this scan found) MaveDB. Any DMS-based validation would have to borrow from a homologous ion channel as a transfer/sanity check, not a direct nAChR benchmark.
5. **Report both a high-specificity and a high-sensitivity operating point**, not a single threshold-free AUC — clinical-triage use and exploratory-research use of a nAChR GOF/LOF score have different tolerances for false positives.
6. **Don't conflate pathogenicity with direction** (the CSH Perspectives paper's central point, reinforced by the PIEZO1 case study): AlphaMissense/REVEL/CADD-style scores tell you "how damaging," not "which direction" — they belong in our pipeline as *inputs*, never as a stand-in for the GOF/LOF label itself.
7. **If we ever release this model**, follow the "Guidelines for releasing a VEP" checklist: publish full training-data provenance, use standard variant representations (HGVS/GA4GH VRS/ClinGen Allele Registry IDs via Ensembl VEP infra), version releases, and report calibrated ACMG-style evidence-strength tiers rather than a bare score.

---
## 5. Updated recommendation (supersedes the June 5-step list where it overlaps)

1. **Keep the nAChR-specific framing** — confirmed novel; no existing tool (MissION, PreMode, or anything found in this scan) covers Cys-loop / pentameric ligand-gated receptors for GOF/LOF direction.
2. **Add cheap population-free scores first** (iGEMME/GEMME, ESCOTT) alongside AlphaMissense — cheaper than a full ESM embedding pipeline and gives a circularity-tier spread (population-free vs. population-tuned) for free. See `features.ipynb`.
3. **Run the PIEZO1-style sanity check early**: pull AlphaMissense/CADD scores for our known GOF/LOF variants and check whether they separate by score + structural region *before* investing in supervised feature engineering — cheap, and tells us how much signal is "free."
4. **Treat the CMS paper's finding as a first-class feature, not an afterthought**: a binding-site-loop-vs-M2-pore-helix domain indicator has direct mechanistic backing for our exact task.
5. **Prototype the MTL phylogeny-kernel idea** (§3.2) once the classical feature set is exhausted — our subunit-imputation problem (§3.2 table) is precisely the uneven-per-gene-data scenario MTL kernels are designed for.
6. **Adopt Ensembl VEP as annotation infrastructure**, and adopt the circularity-tier tagging (§4.2) as a standing convention in every results table from now on.
7. **Longer-horizon, higher-novelty option**: an HPO phenotype-similarity kernel (§3.1) — under-explored even in the direction-of-effect literature, and nAChR phenotypes (CMS, ADNFLE, nicotine dependence) are well-annotated enough to support it.

**Open question to raise with your advisor (new, replaces the June list where resolved):** Livesey & Marsh's circularity-tier taxonomy (population-free / population-tuned / clinical-trained) — should our *own* released model be required to disclose this tier for transparency, the way we're now asking of every input feature?

---
## References and links

*Checked against sources July 2026. As in `vep_research.ipynb`, verify exact author lists/years before formal write-up.*

**Field-wide re-ranking**
- Livesey & Marsh (2025). Pairwise win-rate benchmark of 97 VEPs against DMS data, 36 proteins. *Genome Biology*. (CPT-1, AlphaMissense, ESCOTT, GEMME/iGEMME, popEVE, SaProt, and the circularity/win-rate methodology all from this paper.)
- VARITY — Wu Y, et al. Supervised meta-predictor with heterogeneous-source weighting. Referenced via the DMM 2022 review, Box 3.
- MTBAN — "An enhanced variant effect predictor based on a deep generative model and the Born-Again Networks" (`VEP papers/`).
- Envision — Gray VE, Hause RJ, Luebeck J, Shendure J, Fowler DM (2018). *Cell Systems*.

**Direction-of-effect / ion-channel specific**
- MissION — Martens et al. medRxiv 2025.10.16.25337735; published *Journal of Human Genetics* (2026), DOI 10.1038/s10038-026-01484-9. Tool: https://www.synaptica.nl/variant-interpreter
- PreMode — *Nature Communications* (2025), DOI 10.1038/s41467-025-62318-4.
- Ion-channel phenotypic ML (HPO/MKL) — "Predicting functional effects of ion channel variants using new phenotypic machine learning methods" (`VEP papers/`).
- Voltage-gated K+ channel MTL — *EBioMedicine* (2022) (`VEP papers/`).
- Engel AG, Ohno K, Sine SM (2002). "The Spectrum of Congenital Myasthenic Syndromes." *Molecular Neurobiology* — nAChR CMS mutations, fast- vs slow-channel syndrome mechanisms (`VEP papers/`, filed as "PDF.js viewer.pdf").
- Pillai J, et al. (2025). "Deciphering gain-of-function from loss-of-function variants with AlphaMissense: a case study with... PIEZO1." bioRxiv 2025.07.03.662957. Code: https://github.com/Joshua-Pillai/PIEZO1
- GABA_A receptor + AlphaMissense/Rhapsody — *Israel Journal of Chemistry* (2024), DOI 10.1002/ijch.202300161; follow-up on paralogous GABA_A epilepsy variants, PMC11648851.

**Annotation infrastructure and reviews**
- The Ensembl Variant Effect Predictor — *Genome Biology* (2016), s13059-016-0974-4 (`VEP papers/`).
- Ensembl VEP tutorial — Hunt et al., *Human Mutation* (2021) (`VEP papers/`).
- "Guidelines for releasing a variant effect predictor" (`VEP papers/`).
- "Critical assessment of missense variant effect predictors on disease-relevant variant data" (`VEP papers/`).
- "Benchmarking computational variant effect predictors by their ability to infer human traits" (`VEP papers/`).
- "Variant Effect Prediction in the Age of Machine Learning" — *Cold Spring Harb Perspect* (2024) (`VEP papers/`).
- "VEP Finder" systematic review — 118 tools cataloged (`VEP papers/Review papers/`).
- APF2 — pharmacogenomic ensemble VEP (`VEP papers/`).
- AlphaGenome — DeepMind regulatory-variant DNA model (`VEP papers/`).

**See also:** `vep_research.ipynb` (2026-06-14) for the foundational 5-part-pipeline framing, the six-family map, and the original funNCion/LoGoFunc/PreMode/MissION deep-dives; `NOTES2.ipynb` §9-12 for the current feature set and this project's own prior feature-idea list; `features.ipynb` for concrete implementation recipes for everything proposed here.